In [7]:
import heapq

# Node class (re-used from previous example, slightly adapted for general nodes)
class GraphNode:
    def __init__(self, name, position=None, parent=None):
        self.name = name          # Identifier for the node (e.g., 'A', 'B', (x,y) tuple)
        self.position = position  # Optional: for calculating heuristics if nodes have coordinates
        self.parent = parent
        self.g = float('inf')  # Cost from start to current node
        self.h = float('inf')  # Heuristic (estimated cost from current node to goal)
        self.f = float('inf')  # Total cost (g + h)

    def __eq__(self, other):
        return self.name == other.name

    def __hash__(self):
        return hash(self.name)

    # For use with heapq (priority queue)
    def __lt__(self, other):
        return self.f < other.f

    def __repr__(self):
        return f"GraphNode(name='{self.name}', f={self.f})"

In [8]:
def heuristic_graph(node_a_pos, node_b_pos):
    """Calculates the Euclidean distance heuristic between two node positions."""
    if node_a_pos is None or node_b_pos is None:
        # If positions are not provided, return 0 (effectively Dijkstra's)
        return 0
    return ((node_a_pos[0] - node_b_pos[0])**2 + (node_a_pos[1] - node_b_pos[1])**2)**0.5

def astar_search_graph(graph_adj, start_name, end_name, node_positions=None):

    open_list = []  # Priority queue
    closed_set = set()

    # Dictionary to store the actual Node objects for each node name, to update G values
    # This is important to avoid creating duplicate nodes with potentially higher G values
    # when a shorter path to an already 'opened' node is found.
    node_map = {}

    # Create start node
    start_node = GraphNode(start_name, position=node_positions.get(start_name) if node_positions else None)
    start_node.g = 0
    start_node.h = heuristic_graph(start_node.position, node_positions.get(end_name) if node_positions else None)
    start_node.f = start_node.g + start_node.h
    heapq.heappush(open_list, start_node)
    node_map[start_name] = start_node

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.name in closed_set:
            continue

        closed_set.add(current_node.name)

        if current_node.name == end_name:
            path = []
            current = current_node
            while current is not None:
                path.append(current.name)
                current = current.parent
            return path[::-1]  # Return reversed path

        for neighbor_name, weight in graph_adj.get(current_node.name, []):
            if neighbor_name in closed_set:
                continue

            # Calculate potential g value for the neighbor
            tentative_g = current_node.g + weight

            if neighbor_name not in node_map or tentative_g < node_map[neighbor_name].g:
                neighbor_node = node_map.get(neighbor_name, GraphNode(neighbor_name,
                                                                      position=node_positions.get(neighbor_name) if node_positions else None))
                neighbor_node.parent = current_node
                neighbor_node.g = tentative_g
                neighbor_node.h = heuristic_graph(neighbor_node.position, node_positions.get(end_name) if node_positions else None)
                neighbor_node.f = neighbor_node.g + neighbor_node.h

                if neighbor_name not in node_map:
                    heapq.heappush(open_list, neighbor_node)
                    node_map[neighbor_name] = neighbor_node
                else:
                    # If node is already in open_list, update its priority. Python's heapq doesn't directly support
                    # updating priorities. A common workaround is to push the updated node again,
                    # and rely on the closed_set check to ignore older, higher-cost entries.
                    heapq.heappush(open_list, neighbor_node)

    return None # No path found

In [9]:
road_network = {
    'A': [('B', 6), ('D', 1)],
    'B': [('A', 6), ('E', 2), ('C', 5)],
    'C': [('B', 5), ('F', 5)],
    'D': [('A', 1), ('E', 1)],
    'E': [('B', 2), ('D', 1), ('F', 1)],
    'F': [('C', 5), ('E', 1)]
}

# Define positions for heuristic calculation (e.g., (latitude, longitude) or (x, y))
city_positions = {
    'A': (0, 0),
    'B': (2, 3),
    'C': (5, 4),
    'D': (1, 1),
    'E': (3, 2),
    'F': (6, 1)
}

start_city = 'A'
end_city = 'F'

print(f"Finding path from {start_city} to {end_city} using A*...")
path_found = astar_search_graph(road_network, start_city, end_city, city_positions)

if path_found:
    print(f"Shortest path: {path_found}")
else:
    print(f"No path found from {start_city} to {end_city}.")

# Example without positions (heuristic will always be 0, like Dijkstra's)
print("\nFinding path without position-based heuristic (effectively Dijkstra's):")
path_found_no_heuristic = astar_search_graph(road_network, start_city, end_city)

if path_found_no_heuristic:
    print(f"Shortest path: {path_found_no_heuristic}")
else:
    print(f"No path found from {start_city} to {end_city}.")

Finding path from A to F using A*...
Shortest path: ['A', 'D', 'E', 'F']

Finding path without position-based heuristic (effectively Dijkstra's):
Shortest path: ['A', 'D', 'E', 'F']
